# Multi-Agent Agentic RAG with CrewAI

This notebook implements a multi-agent Retrieval-Augmented Generation system with:
- **Router Agent**: classifies each user question to either PDF retrieval or web retrieval
- **Retriever Agent**: executes the selected retrieval path and returns a source-grounded answer

It supports a PDF vector search path (FAISS) and a web search path (Tavily), with execution tracing and visualization.

In [1]:
import os
import re
import json
import time
import logging
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Literal

import requests
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai.tools import BaseTool

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import AzureOpenAIEmbeddings, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

warnings.filterwarnings('ignore', category=DeprecationWarning)
PROJECT_ROOT = Path('/home/asima.khalid557/crewai-multi-rag')
load_dotenv(PROJECT_ROOT / '.env', override=True)

/tmp/ipykernel_23236/3818975386.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

## 1) Environment and Logging

This notebook loads variables from a project-level `.env` file.

Required keys:
- `AZURE_OPENAI_API_KEY` and `AZURE_OPENAI_ENDPOINT` (or `OPENAI_API_KEY`)
- `AZURE_EMBEDDING_DEPLOYMENT` (if using Azure embeddings)
- `TAVILY_API_KEY` for web search

In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

PDF_PATH = os.getenv('PDF_PATH', 'research_paper.pdf')

def check_required_env():
    has_azure = bool(os.getenv('AZURE_OPENAI_API_KEY') and os.getenv('AZURE_OPENAI_ENDPOINT'))
    has_openai = bool(os.getenv('OPENAI_API_KEY'))

    if not (has_azure or has_openai):
        raise ValueError('Set AZURE_OPENAI_API_KEY + AZURE_OPENAI_ENDPOINT OR OPENAI_API_KEY')

    # Web key is only required when web retrieval route is actually used.
    if not os.getenv('TAVILY_API_KEY'):
        logging.warning('TAVILY_API_KEY is not set. Web route may fail if selected.')

check_required_env()
print('Environment looks good.')

Environment looks good.


In [3]:
def _env_int(name: str, default: int) -> int:
    try:
        return int(os.getenv(name, str(default)))
    except (TypeError, ValueError):
        return default

def build_llm() -> LLM:
    azure_key = os.getenv('AZURE_OPENAI_API_KEY')
    azure_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')

    if azure_key and azure_endpoint:
        return LLM(
            model=os.getenv('AZURE_CHAT_MODEL', 'azure/gpt-4o-mini'),
            api_key=azure_key,
            base_url=azure_endpoint.rstrip('/') + '/',
            api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-12-01-preview'),
            is_litellm=True,
            temperature=0.2,
        )

    return LLM(
        model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'),
        api_key=os.getenv('OPENAI_API_KEY'),
        temperature=0.2,
    )

def build_embeddings():
    azure_key = os.getenv('AZURE_OPENAI_API_KEY')
    azure_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
    azure_deployment = os.getenv('AZURE_EMBEDDING_DEPLOYMENT')
    embedding_batch_size = max(1, _env_int('EMBEDDING_BATCH_SIZE', 1))

    if azure_key and azure_endpoint and azure_deployment:
        return AzureOpenAIEmbeddings(
            api_key=azure_key,
            azure_endpoint=azure_endpoint,
            api_version=os.getenv('AZURE_EMBEDDING_API_VERSION', '2023-05-15'),
            deployment=azure_deployment,
            chunk_size=embedding_batch_size,
        )

    return OpenAIEmbeddings(
        api_key=os.getenv('OPENAI_API_KEY'),
        model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'),
        chunk_size=embedding_batch_size,
    )

In [4]:
@dataclass
class RouteDecision:
    route: Literal['pdf', 'web']
    reason: str

def build_pdf_retriever(pdf_path: str):
    import hashlib

    pdf_file = Path(pdf_path)
    if not pdf_file.exists():
        raise FileNotFoundError(f'PDF file not found: {pdf_file}')

    loader = PyPDFLoader(str(pdf_file))
    docs = loader.load()

    max_pages = max(0, _env_int('PDF_MAX_PAGES', 8))
    if max_pages > 0:
        docs = docs[:max_pages]

    splitter = CharacterTextSplitter(
        chunk_size=max(200, _env_int('RAG_CHUNK_SIZE', 500)),
        chunk_overlap=max(0, _env_int('RAG_CHUNK_OVERLAP', 80)),
    )
    chunks = splitter.split_documents(docs)

    max_chunks = max(0, _env_int('RAG_MAX_CHUNKS', 60))
    if max_chunks > 0:
        chunks = chunks[:max_chunks]

    if not chunks:
        raise ValueError('No text chunks were created from the PDF.')

    cache_root = PROJECT_ROOT / 'faiss_cache'
    cache_root.mkdir(parents=True, exist_ok=True)

    cache_key = f"{pdf_file.resolve()}::{pdf_file.stat().st_mtime_ns}::{len(chunks)}"
    cache_name = hashlib.sha1(cache_key.encode('utf-8')).hexdigest()[:16]
    cache_path = cache_root / cache_name

    embeddings = build_embeddings()
    retrieval_k = max(1, _env_int('RAG_TOP_K', 4))

    if cache_path.exists():
        vectorstore = FAISS.load_local(
            str(cache_path),
            embeddings,
            allow_dangerous_deserialization=True,
        )
        return vectorstore.as_retriever(search_kwargs={'k': retrieval_k})

    try:
        vectorstore = FAISS.from_documents(chunks, embeddings)
    except Exception as exc:
        err = str(exc)
        if '429' in err or 'RateLimitError' in err or 'Token limit' in err:
            raise RuntimeError(
                'Embedding API rate limit hit (429). Reduce load using PDF_MAX_PAGES, RAG_MAX_CHUNKS, '
                'or EMBEDDING_BATCH_SIZE, then retry.'
            ) from exc
        raise

    vectorstore.save_local(str(cache_path))
    return vectorstore.as_retriever(search_kwargs={'k': retrieval_k})

def create_pdf_tool(retriever):
    class PDFVectorSearchTool(BaseTool):
        name: str = 'pdf_vector_search'
        description: str = 'Retrieve answer evidence from PDF vector search.'

        def _run(self, query: str) -> str:
            docs = retriever.invoke(query)
            lines = []
            for idx, d in enumerate(docs, start=1):
                page = d.metadata.get('page', '?')
                content = d.page_content.strip().replace('\n', ' ')
                lines.append(f'[PDF-{idx} | page {page}] {content[:500]}')
            return '\n\n'.join(lines)[:3000]

    return PDFVectorSearchTool()

class WebSearchTool(BaseTool):
    name: str = 'web_search'
    description: str = 'Search the web using Tavily and return source snippets.'

    def _run(self, query: str) -> str:
        payload = {
            'api_key': os.getenv('TAVILY_API_KEY'),
            'query': query,
            'max_results': 5,
        }
        response = requests.post('https://api.tavily.com/search', json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

        rows = []
        for idx, item in enumerate(data.get('results', []), start=1):
            rows.append(
                f"[WEB-{idx}] {item.get('title', 'Untitled')} | {item.get('url', '')}\n{item.get('content', '')}"
            )
        return '\n\n'.join(rows)[:3500]

retriever = build_pdf_retriever(PDF_PATH)
pdf_tool = create_pdf_tool(retriever)
web_tool = WebSearchTool()
llm = build_llm()
print('Tools and models are ready.')

2026-06-21 13:13:43,755 - INFO - Retrying request to /embeddings in 56.000000 seconds
2026-06-21 13:14:40,504 - INFO - Loading faiss with AVX2 support.
2026-06-21 13:14:40,505 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-06-21 13:14:40,505 - INFO - Loading faiss.
2026-06-21 13:14:40,531 - INFO - Successfully loaded faiss.


Tools and models are ready.


In [ ]:
import asyncio

def parse_route_output(raw_output: str, question: str) -> RouteDecision:
    match = re.search(r'\{[\s\S]*\}', raw_output)
    if match:
        try:
            payload = json.loads(match.group(0))
            route = str(payload.get('route', '')).lower().strip()
            reason = str(payload.get('reason', '')).strip() or 'No reason provided'
            if route in {'pdf', 'web'}:
                return RouteDecision(route=route, reason=reason)
        except json.JSONDecodeError:
            pass

    q = question.lower()
    pdf_keywords = ['document', 'paper', 'section', 'method', 'dataset', 'experiment', 'internal']
    fallback = 'pdf' if any(k in q for k in pdf_keywords) else 'web'
    return RouteDecision(route=fallback, reason='Fallback keyword route due to invalid router output')

async def run_router_async(question: str, llm: LLM, trace: List[Dict[str, str]]) -> RouteDecision:
    router = Agent(
        role='Router Agent',
        goal="Classify each question to either 'pdf' or 'web' retrieval path.",
        backstory='You route research and document QA queries based on source suitability.',
        llm=llm,
        allow_delegation=False,
        verbose=True,
    )

    task = Task(
        description=(
            'Return ONLY JSON in this schema: {\"route\": \"pdf|web\", \"reason\": \"short reason\"}.\n'
            f'Question: {question}'
        ),
        expected_output='Strict JSON object with route and reason.',
        agent=router,
    )

    t0 = time.perf_counter()
    raw = str(await Crew(agents=[router], tasks=[task], process=Process.sequential, verbose=True).kickoff_async())
    ms = int((time.perf_counter() - t0) * 1000)

    decision = parse_route_output(raw, question)
    trace.append({'step': 'router', 'route': decision.route, 'reason': decision.reason, 'duration_ms': str(ms)})
    return decision

async def run_retriever_async(
    question: str,
    decision: RouteDecision,
    llm: LLM,
    pdf_tool: BaseTool,
    web_tool: BaseTool,
    trace: List[Dict[str, str]],
) -> str:
    selected_tool = pdf_tool if decision.route == 'pdf' else web_tool

    retriever_agent = Agent(
        role='Retriever Agent',
        goal='Use the selected source tool and answer with evidence-backed output.',
        backstory='You are an evidence-grounded retrieval and synthesis specialist.',
        tools=[selected_tool],
        llm=llm,
        allow_delegation=False,
        verbose=True,
    )

    task = Task(
        description=(
            f'Question: {question}\n'
            f'Route: {decision.route}\n'
            f'Reason: {decision.reason}\n\n'
            'Instructions:\n'
            '1) Use exactly one available tool.\n'
            '2) Produce a concise answer from retrieved evidence only.\n'
            '3) End with a Sources section.'
        ),
        expected_output='A concise final answer and Sources section.',
        agent=retriever_agent,
    )

    t0 = time.perf_counter()
    answer = str(await Crew(agents=[retriever_agent], tasks=[task], process=Process.sequential, verbose=True).kickoff_async())
    ms = int((time.perf_counter() - t0) * 1000)

    trace.append({'step': 'retriever', 'route': decision.route, 'reason': f'Used tool: {selected_tool.name}', 'duration_ms': str(ms)})
    return answer

async def run_pipeline_async(question: str, pdf_path: str = PDF_PATH):
    local_trace: List[Dict[str, str]] = []

    decision = await run_router_async(question, llm, local_trace)
    answer = await run_retriever_async(question, decision, llm, pdf_tool, web_tool, local_trace)

    return {
        'question': question,
        'route': decision.route,
        'route_reason': decision.reason,
        'answer': answer,
        'trace': local_trace,
    }

def run_pipeline(question: str, pdf_path: str = PDF_PATH):
    # Convenience wrapper for scripts; in notebooks prefer: await run_pipeline_async(...)
    try:
        asyncio.get_running_loop()
        raise RuntimeError('Running event loop detected. In notebook use: await run_pipeline_async(query, PDF_PATH)')
    except RuntimeError as exc:
        if 'no running event loop' in str(exc).lower():
            return asyncio.run(run_pipeline_async(question, pdf_path))
        raise

In [ ]:
# Example queries
query = 'Summarize the key findings of this research paper.'
# query = 'What are the latest updates in retrieval-augmented generation methods?'

result = await run_pipeline_async(query, PDF_PATH)

print('Routing decision:', result['route'])
print('Routing reason:', result['route_reason'])
print('\nFinal answer:\n')
print(result['answer'])

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 35b6c18e-62cb-4108-803f-516260f07ed6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Return ONLY JSON in this schema: {"route": "pdf|web", "reason": "short reason"}.                         │
│  Question: Summarize the key findings of this research paper.                                                   │
│  ID: 1f20d85d-b788-44e1-b878-b92234d8b31e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Return ONLY JSON in this schema: {"route": "pdf|web", "reason": "short reason"}.                         │
│  Question: Summarize the key findings of this research paper.                                                   │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 35b6c18e-62cb-4108-803f-516260f07ed6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.

In [ ]:
trace_df = pd.DataFrame(result['trace'])
trace_df['duration_ms'] = trace_df['duration_ms'].astype(int)
display(trace_df)

plt.figure(figsize=(7, 4))
plt.bar(trace_df['step'], trace_df['duration_ms'])
plt.title('Agent Reasoning Trace (Execution Time by Step)')
plt.ylabel('Duration (ms)')
plt.xlabel('Pipeline Step')
plt.tight_layout()
plt.show()

## Result Checklist

- Router agent classifies queries into `pdf` or `web`
- Retriever agent uses the selected retrieval method
- Final answer is source-grounded
- Orchestration and step trace are logged and visualized